In [1]:
import networkx as nx
import pandas as pd
import numpy as np

import src

In [2]:
path = src.PATH / "data/clean/networks/"
graph_files = list(path.iterdir())

In [3]:
sentiment_path = src.PATH / "data/interim/title_sentiments"

sentiment_df = pd.DataFrame()

for file in sentiment_path.iterdir():
    temp = pd.read_csv(file)
    sentiment_df = pd.concat([sentiment_df, temp], axis=0)

sentiment_dict = (
    sentiment_df.drop_duplicates(subset=["video_id"])
    .set_index("video_id", drop=True)
    .to_dict(orient="index")
)

In [4]:
sentiment_path = src.PATH / "data/interim/perspective_data"

perspective_df = pd.DataFrame()

for file in sentiment_path.iterdir():
    temp = pd.read_csv(file)
    perspective_df = pd.concat([perspective_df, temp], axis=0)

perspective_dict = (
    perspective_df.drop_duplicates(subset=["video_id"])
    .set_index("video_id", drop=True)
    .to_dict(orient="index")
)

In [5]:
leza_path = src.PATH / "data/external/recfluence_channel_review.csv"

leza_df = pd.read_csv(leza_path)

leza_dict = (
    leza_df[["CHANNEL_ID", "TAGS", "LR"]]
    .drop_duplicates()
    .set_index("CHANNEL_ID", drop=True)
    .to_dict(orient="index")
)

In [6]:
for file in graph_files:
    g = nx.read_gml(file)

    for node, data in g.nodes(data=True):
        val = sentiment_dict.get(node, {}).get("sentiment", np.nan)
        data["sentiment"] = val

        val = perspective_dict.get(node, {})
        for k, v in val.items():
            data[k] = v

        val = leza_dict.get(data["channelid"], {}).get("LR", "")
        data["leftright"] = val

        val = leza_dict.get(data["channelid"], {}).get("TAGS", "")
        data["channeltags"] = val

    nx.write_gml(g, file)

KeyError: 'channel_id'